# 🚀 ISIRI 2.0 — ByT5-Small Fine-Tuning for Tulu ↔ English Neural Translation

This notebook fine-tunes **`google/byt5-small`** (token-free byte-level Seq2Seq transformer) for bidirectional translation:
- **Task 1 (Tulu ➔ English)**: Spoken/transcribed Tulu command to canonical English intent.
- **Task 2 (English ➔ Tulu)**: English assistant response into Romanised Tulu.

### Why ByT5?
- Handles spelling variations (*malpule, malpu, malpulet*) with **zero `<UNK>` errors**.
- 100% token-free byte vocabulary.

## 1. Install Dependencies & Check GPU

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu sacremoses sentencepiece accelerate

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Upload and Prepare Dataset

In [ ]:
import os
import pandas as pd
from google.colab import files
from datasets import Dataset, DatasetDict

DATASET_FILE = "clean_dataset.csv"

if not os.path.exists(DATASET_FILE):
    print("Please upload your 'clean_dataset.csv' file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".csv"):
            DATASET_FILE = fn
            break

df = pd.read_csv(DATASET_FILE)
df = df.dropna(subset=["English", "Tulu"])
df["English"] = df["English"].astype(str).str.strip()
df["Tulu"] = df["Tulu"].astype(str).str.strip()
df = df[(df["English"] != "") & (df["Tulu"] != "")]

print(f"Loaded {len(df)} parallel pairs.")

# Build Bidirectional Dataset
inputs, targets, directions = [], [], []
for _, row in df.iterrows():
    en, tu = row["English"], row["Tulu"]
    # Tulu -> English
    inputs.append(f"translate Tulu to English: {tu}")
    targets.append(en)
    directions.append("tulu_to_en")
    # English -> Tulu
    inputs.append(f"translate English to Tulu: {en}")
    targets.append(tu)
    directions.append("en_to_tulu")

raw_dataset = Dataset.from_dict({
    "input_text": inputs,
    "target_text": targets,
    "direction": directions
})

train_test = raw_dataset.train_test_split(test_size=0.15, seed=42)
test_valid = train_test["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": train_test["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

print(f"Dataset Splits: Train={len(dataset_dict['train'])}, Val={len(dataset_dict['validation'])}, Test={len(dataset_dict['test'])}")

## 3. Tokenize with ByT5 Byte-Level Tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_SOURCE_LENGTH = 160
MAX_TARGET_LENGTH = 160

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text", "direction"]
)
print("Tokenization complete!")

## 4. Setup Model, Metrics & Seq2SeqTrainer

In [ ]:
import numpy as np
import torch
import evaluate
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

# Direct safe byte decoding for ByT5
def decode_byt5_sequences(sequences):
    decoded = []
    for seq in sequences:
        raw_bytes = bytearray([int(t) - 3 for t in seq if 3 <= int(t) <= 258])
        decoded.append(raw_bytes.decode("utf-8", errors="ignore").strip())
    return decoded

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    decoded_preds = decode_byt5_sequences(preds)
    decoded_labels = [[l] for l in decode_byt5_sequences(labels)]
    
    bleu = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"]
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"]
    
    return {
        "bleu": round(bleu, 2),
        "chrf": round(chrf, 2)
    }

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

OUTPUT_DIR = "./byt5_checkpoints"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=12,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=2,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    load_best_model_at_end=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

## 5. Train Model

In [ ]:
print("Starting training on GPU...")
train_output = trainer.train()
print("Training finished successfully!")
print(train_output)

## 6. Evaluate on Test Set

In [ ]:
test_results = trainer.predict(
    test_dataset=tokenized_datasets["test"],
    metric_key_prefix="test"
)
print(f"\n=== TEST SET PERFORMANCE ===")
print(f"Test SacreBLEU: {test_results.metrics.get('test_bleu')}")
print(f"Test chrF++:   {test_results.metrics.get('test_chrf')}")

## 7. Interactive Translation Playground

In [ ]:
def translate(text, direction="tulu_to_en"):
    prefix = "translate Tulu to English: " if direction == "tulu_to_en" else "translate English to Tulu: "
    prompt = prefix + text
    inputs = tokenizer(prompt, return_tensors="pt", max_length=128, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=48,
            num_beams=3,
            early_stopping=True,
            decoder_start_token_id=0,
            eos_token_id=1,
            pad_token_id=0
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Test sample translations
test_samples = [
    ("Open YouTube", "en_to_tulu"),
    ("Turn on the bedroom light", "en_to_tulu"),
    ("Turn off the fan", "en_to_tulu"),
    ("How are you?", "en_to_tulu"),
    ("youtube open malpule", "tulu_to_en"),
    ("kone da light on malpule", "tulu_to_en"),
    ("ini mangalore da weather encha undu", "tulu_to_en"),
    ("yaan illag povond ulle", "tulu_to_en"),
]

print("\n=== SAMPLE INFERENCE VERIFICATION ===")
for sentence, direction in test_samples:
    result = translate(sentence, direction)
    print(f"[{direction}] '{sentence}' -> '{result}'")

## 8. Export Model for ISIRI 2.0

In [ ]:
FINAL_EXPORT = "./byt5_tulu_english"
os.makedirs(FINAL_EXPORT, exist_ok=True)

# Direct save of fine-tuned model weights
model.save_pretrained(FINAL_EXPORT)
tokenizer.save_pretrained(FINAL_EXPORT)

# Zip the model for download
!zip -r byt5_tulu_english.zip ./byt5_tulu_english

print("\n✅ Trained model saved directly and zipped as 'byt5_tulu_english.zip'!")

# Trigger browser download
try:
    files.download('byt5_tulu_english.zip')
except Exception as e:
    print("Download trigger notice:", e)